# 65 - Late Fusion TL across All Benchmarks

Menambahkan **Late Fusion TL** (ResNet18 CNN_TL + FCNN) yang sebelumnya terlewat di benchmark notebooks (36, 37, 60, 61, 62).

**Strategi**:
1. Reuse checkpoint `CNN_TL_B1/model.pth` dan `FCNN_B1/model.pth` kalau sudah ada
2. Train dari nol kalau belum ada (fallback)
3. Late Fusion = weighted softmax averaging (tune weight di val, eval di test)
4. Bonus: fill missing `micro_f1` = `accuracy` di JSON existing (multi-class single-label: identical)

**Output**: update `{dataset}_{num_class}c_results.json` dengan key `Late_Fusion_TL_B1`.

**Datasets**: CK+ 7c/4c, JAFFE 7c/4c, RAF-DB 7c/4c, KDEF 7c/4c, Primer 7c/4c — **10 benchmark variants**.

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionCNNTransfer, EmotionFCNN
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
LR_TL = 0.00005
LR = 0.0001

BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
MODELS_DIR = PROJECT_ROOT / 'models' / 'benchmark'

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

Device: cuda
Setup complete.


In [2]:
# ── Helpers ──

def _subject_split(subjects, seed=42, train_ratio=0.8, val_ratio=0.1):
    rng = np.random.RandomState(seed)
    uniq = np.array(sorted(set(subjects.tolist())))
    rng.shuffle(uniq)
    n = len(uniq)
    n_tr = int(n * train_ratio)
    n_v = int(n * val_ratio)
    return set(uniq[:n_tr].tolist()), set(uniq[n_tr:n_tr+n_v].tolist()), set(uniq[n_tr+n_v:].tolist())


def load_dataset(dataset_name, num_classes):
    """Return (tr_img, tr_lm, tr_y, v_img, v_lm, v_y, te_img, te_lm, te_y)."""
    if dataset_name == 'rafdb':
        d = BENCHMARK_DIR / f'rafdb_{num_classes}class'
        X = np.load(d / 'X_train_images.npy')
        L = np.load(d / 'X_train_landmarks.npy')
        y = np.load(d / 'y_train.npy')
        te_X = np.load(d / 'X_test_images.npy')
        te_L = np.load(d / 'X_test_landmarks.npy')
        te_y = np.load(d / 'y_test.npy')
        idx_tr, idx_v = train_test_split(np.arange(len(y)), test_size=0.1, stratify=y, random_state=42)
        return (X[idx_tr], L[idx_tr], y[idx_tr],
                X[idx_v], L[idx_v], y[idx_v],
                te_X, te_L, te_y)

    if dataset_name == 'kdef':
        d = BENCHMARK_DIR / f'kdef_{num_classes}class'
        return (np.load(d / 'X_train_images.npy'),
                np.load(d / 'X_train_landmarks.npy'),
                np.load(d / 'y_train.npy'),
                np.load(d / 'X_val_images.npy'),
                np.load(d / 'X_val_landmarks.npy'),
                np.load(d / 'y_val.npy'),
                np.load(d / 'X_test_images.npy'),
                np.load(d / 'X_test_landmarks.npy'),
                np.load(d / 'y_test.npy'))

    if dataset_name == 'primer':
        tr_X = np.load(PRIMER_DIR / 'X_train_images.npy')
        tr_L = np.load(PRIMER_DIR / 'X_train_landmarks.npy')
        tr_y = np.load(PRIMER_DIR / 'y_train.npy')
        v_X = np.load(PRIMER_DIR / 'X_val_images.npy')
        v_L = np.load(PRIMER_DIR / 'X_val_landmarks.npy')
        v_y = np.load(PRIMER_DIR / 'y_val.npy')
        te_X = np.load(PRIMER_DIR / 'X_test_images.npy')
        te_L = np.load(PRIMER_DIR / 'X_test_landmarks.npy')
        te_y = np.load(PRIMER_DIR / 'y_test.npy')
        if num_classes == 4:
            tr_y = REMAP_4[tr_y]
            v_y = REMAP_4[v_y]
            te_y = REMAP_4[te_y]
        return tr_X, tr_L, tr_y, v_X, v_L, v_y, te_X, te_L, te_y

    if dataset_name in ('ckplus', 'jaffe'):
        # Old-style benchmark: X_images.npy, subjects.npy (single split file)
        if dataset_name == 'ckplus' and num_classes == 4:
            d = BENCHMARK_DIR / 'ckplus_4class_contempt'  # includes contempt
        else:
            d = BENCHMARK_DIR / f'{dataset_name}_{num_classes}class'
        X = np.load(d / 'X_images.npy')
        L = np.load(d / 'X_landmarks.npy')
        y = np.load(d / 'y_labels.npy')
        subjects = np.load(d / 'subjects.npy', allow_pickle=True)
        tr_subs, v_subs, te_subs = _subject_split(subjects)
        tr_idx = np.where(np.isin(subjects, list(tr_subs)))[0]
        v_idx = np.where(np.isin(subjects, list(v_subs)))[0]
        te_idx = np.where(np.isin(subjects, list(te_subs)))[0]
        return (X[tr_idx], L[tr_idx], y[tr_idx],
                X[v_idx], L[v_idx], y[v_idx],
                X[te_idx], L[te_idx], y[te_idx])

    raise ValueError(f'Unknown dataset {dataset_name}')


def checkpoint_dir(dataset_name, num_classes, model_key):
    """Checkpoint path. Handles naming convention differences:
    - rafdb/kdef/primer: models/benchmark/{ds}/{num}c/{key}/
    - ckplus/jaffe:      models/benchmark/{ds}/{ds}_{num}c/{key}/  (nb 36/37 style)
    """
    if dataset_name in ('ckplus', 'jaffe'):
        return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c' / model_key
    return MODELS_DIR / dataset_name / f'{num_classes}c' / model_key


def results_file(dataset_name, num_classes):
    return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c_results.json'


def make_loader(img, lm, y, model_type, batch_size=BATCH_SIZE, shuffle=True):
    img_t = torch.from_numpy(img).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(lm)
    y_t = torch.from_numpy(y).long()
    if model_type == 'cnn':
        ds = TensorDataset(img_t, y_t)
    else:
        ds = TensorDataset(lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def ensure_checkpoint(dataset_name, num_classes, model_key, model_class, model_type, lr,
                     tr_img, tr_lm, tr_y, v_img, v_lm, v_y):
    save_dir = checkpoint_dir(dataset_name, num_classes, model_key)
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / 'model.pth'

    model = model_class(num_classes=num_classes).to(device)
    if save_path.exists():
        print(f'    [SKIP train] loading existing {save_path}')
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
        return model

    print(f'    [TRAIN] {model_key} ...')
    tr_loader = make_loader(tr_img, tr_lm, tr_y, model_type, BATCH_SIZE, True)
    val_loader = make_loader(v_img, v_lm, v_y, model_type, BATCH_SIZE, False)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(model, tr_loader, val_loader, crit, opt, sch,
                device, model_type, EPOCHS, PATIENCE, str(save_path))
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    return model


def metrics_triple(y_true, y_pred):
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


@torch.no_grad()
def batched_softmax(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        x = batch[0].to(device)
        probs.append(torch.softmax(model(x), dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)


def fill_missing_micro_f1(results_path):
    """For multi-class single-label, micro_f1 = accuracy.
    Fill it in JSON for entries missing `micro_f1` but having `accuracy`.
    """
    if not results_path.exists():
        return 0
    with open(results_path) as f:
        data = json.load(f)
    updated = 0
    for key, r in data.items():
        if isinstance(r, dict) and 'accuracy' in r and 'micro_f1' not in r:
            r['micro_f1'] = r['accuracy']
            updated += 1
    if updated > 0:
        with open(results_path, 'w') as f:
            json.dump(data, f, indent=2)
    return updated


def late_fusion_tl(dataset_name, num_classes):
    print(f"\n{'='*70}")
    print(f'  Late Fusion TL — {dataset_name.upper()} {num_classes}c')
    print(f"{'='*70}")

    # Fill missing micro_f1 first (for existing 5 models in this JSON)
    rf = results_file(dataset_name, num_classes)
    n_filled = fill_missing_micro_f1(rf)
    if n_filled:
        print(f'  [micro_f1 fill] filled {n_filled} missing entries in {rf.name}')

    tr_img, tr_lm, tr_y, v_img, v_lm, v_y, te_img, te_lm, te_y = load_dataset(dataset_name, num_classes)
    print(f'  Train={len(tr_y)}  Val={len(v_y)}  Test={len(te_y)}')

    cnn_tl = ensure_checkpoint(dataset_name, num_classes, 'CNN_TL_B1',
                                EmotionCNNTransfer, 'cnn', LR_TL,
                                tr_img, tr_lm, tr_y, v_img, v_lm, v_y)
    fcnn = ensure_checkpoint(dataset_name, num_classes, 'FCNN_B1',
                              EmotionFCNN, 'fcnn', LR,
                              tr_img, tr_lm, tr_y, v_img, v_lm, v_y)

    v_cnn_loader = make_loader(v_img, v_lm, v_y, 'cnn', BATCH_SIZE, False)
    v_fcnn_loader = make_loader(v_img, v_lm, v_y, 'fcnn', BATCH_SIZE, False)
    te_cnn_loader = make_loader(te_img, te_lm, te_y, 'cnn', BATCH_SIZE, False)
    te_fcnn_loader = make_loader(te_img, te_lm, te_y, 'fcnn', BATCH_SIZE, False)

    vc = batched_softmax(cnn_tl, v_cnn_loader)
    vf = batched_softmax(fcnn, v_fcnn_loader)
    cp = batched_softmax(cnn_tl, te_cnn_loader)
    fp = batched_softmax(fcnn, te_fcnn_loader)

    best_wf1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        pr = (w * vc + (1 - w) * vf).argmax(axis=1)
        f = f1_score(v_y, pr, average='macro', zero_division=0)
        if f > best_wf1:
            best_wf1, best_w = f, w
    print(f'    Best w(CNN_TL)={best_w:.2f}  val macro F1={best_wf1:.4f}')

    preds = (best_w * cp + (1 - best_w) * fp).argmax(axis=1)
    m = metrics_triple(te_y, preds)
    m['best_cnn_tl_weight'] = float(best_w)
    print(f"    Macro={m['macro_f1']:.4f}  Micro={m['micro_f1']:.4f}  Weighted={m['weighted_f1']:.4f}")

    # Update results JSON
    existing = {}
    if rf.exists():
        with open(rf) as f:
            existing = json.load(f)
    existing['Late_Fusion_TL_B1'] = m
    rf.parent.mkdir(parents=True, exist_ok=True)
    with open(rf, 'w') as f:
        json.dump(existing, f, indent=2)
    print(f'    Updated: {rf}')
    return m


print('Helpers ready.')

Helpers ready.


## Run Late Fusion TL across 5 Datasets × 2 Class Configs

In [3]:
all_results = {}

# Primer (paling penting)
all_results['primer_7c'] = late_fusion_tl('primer', 7)
all_results['primer_4c'] = late_fusion_tl('primer', 4)


  Late Fusion TL — PRIMER 7c


  Train=5287  Val=579  Test=929


    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/7c/CNN_TL_B1/model.pth
    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/7c/FCNN_B1/model.pth


    Best w(CNN_TL)=0.25  val macro F1=0.2695
    Macro=0.2851  Micro=0.8267  Weighted=0.8278
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/primer_7c_results.json



  Late Fusion TL — PRIMER 4c


  Train=5287  Val=579  Test=929


    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/4c/CNN_TL_B1/model.pth
    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/4c/FCNN_B1/model.pth


    Best w(CNN_TL)=0.40  val macro F1=0.4884
    Macro=0.4723  Micro=0.7772  Weighted=0.7787
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/primer/primer_4c_results.json


In [4]:
# CK+
all_results['ckplus_7c'] = late_fusion_tl('ckplus', 7)
all_results['ckplus_4c'] = late_fusion_tl('ckplus', 4)


  Late Fusion TL — CKPLUS 7c
  [micro_f1 fill] filled 6 missing entries in ckplus_7c_results.json


  Train=508  Val=69  Test=59


    [TRAIN] CNN_TL_B1 ...
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.9699     0.1949     1.7149    0.3623   0.2371   0.000050  (2.1s)


     2      1.4871     0.5118     1.4426    0.5507   0.4242   0.000050  (1.8s)


...

    34      0.6942     0.7519     0.6689    0.8194   0.6483   0.000100  (0.1s)
    35      0.7042     0.7442     0.6663    0.7500   0.5502   0.000100  (0.1s)
    36      0.6828     0.7423     0.7475    0.7361   0.5438   0.000100  (0.1s)
    37      0.6954     0.7404     0.7096    0.7361   0.5300   0.000100  (0.1s)


    38      0.7017     0.7462     0.7236    0.7778   0.5760   0.000100  (0.1s)
    39      0.6471     0.7692     0.8268    0.6806   0.4930   0.000100  (0.1s)
    40      0.6659     0.7596     0.5796    0.8194   0.6261   0.000100  (0.1s)
    41      0.6605     0.7731     0.7851    0.6250   0.5071   0.000100  (0.1s)


    42      0.6739     0.7635     0.6353    0.7500   0.5575   0.000100  (0.1s)
    43      0.6508     0.7615     0.6693    0.7639   0.5706   0.000100  (0.1s)
    44      0.6591     0.7538     0.6858    0.7500   0.5659   0.000050  (0.1s)
    45      0.6441     0.7769     0.6247    0.7917   0.6236   0.000050  (0.1s)


    46      0.6895     0.7500     0.6243    0.7639   0.5706   0.000050  (0.1s)
    47      0.6799     0.7577     0.5613    0.8056   0.6068   0.000050  (0.1s)
    48      0.6427     0.7750     0.6554    0.7361   0.5522   0.000050  (0.1s)
    49      0.6393     0.7558     0.5818    0.7639   0.5634   0.000050  (0.1s)

Early stopping at epoch 49. Best epoch: 34 (val_f1=0.6483)

Best: epoch 34, val_acc=0.8194, val_f1=0.6483
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/ckplus/ckplus_4c/FCNN_B1/model.pth


    Best w(CNN_TL)=0.90  val macro F1=0.8200
    Macro=0.6039  Micro=0.8065  Weighted=0.8034
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/ckplus/ckplus_4c_results.json


In [5]:
# JAFFE
all_results['jaffe_7c'] = late_fusion_tl('jaffe', 7)
all_results['jaffe_4c'] = late_fusion_tl('jaffe', 4)


  Late Fusion TL — JAFFE 7c
  [micro_f1 fill] filled 6 missing entries in jaffe_7c_results.json


  Train=173  Val=20  Test=20
    [TRAIN] CNN_TL_B1 ...
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      2.0317     0.2023     2.0310    0.1500   0.0373   0.000050  (0.6s)


     2      1.3949     0.5896     2.0462    0.1500   0.0373   0.000050  (0.6s)


     3      1.0835     0.7861     2.0818    0.1500   0.0373   0.000050  (0.6s)


...

     3      1.4501     0.2254     1.4648    0.1500   0.0652   0.000100  (0.0s)
     4      1.4372     0.3121     1.4966    0.1500   0.0652   0.000100  (0.0s)
     5      1.3866     0.3006     1.5097    0.1500   0.0652   0.000100  (0.0s)
     6      1.3744     0.3064     1.5263    0.1500   0.0652   0.000100  (0.0s)
     7      1.3063     0.3931     1.5444    0.1500   0.0652   0.000100  (0.0s)
     8      1.2672     0.4220     1.5478    0.1500   0.0652   0.000100  (0.0s)
     9      1.2461     0.4393     1.5531    0.1500   0.0652   0.000100  (0.0s)
    10      1.2456     0.4740     1.5928    0.2000   0.1504   0.000100  (0.0s)
    11      1.2024     0.4798     1.6015    0.1500   0.0682   0.000100  (0.0s)


    12      1.1999     0.4855     1.5848    0.2000   0.1583   0.000100  (0.0s)
    13      1.1741     0.4971     1.5257    0.2000   0.1932   0.000100  (0.0s)
    14      1.2089     0.4624     1.4705    0.2000   0.1066   0.000100  (0.0s)
    15      1.1219     0.5260     1.4359    0.2500   0.1892   0.000100  (0.0s)
    16      1.0924     0.5723     1.4004    0.1500   0.0973   0.000100  (0.0s)
    17      1.1015     0.5607     1.3778    0.3000   0.3381   0.000100  (0.0s)
    18      1.0431     0.6301     1.4028    0.3000   0.3377   0.000100  (0.0s)


    19      1.0621     0.6069     1.4268    0.1500   0.0885   0.000100  (0.0s)
    20      1.0421     0.6012     1.4122    0.1500   0.0885   0.000100  (0.0s)
    21      1.0457     0.5954     1.3938    0.2000   0.1565   0.000100  (0.0s)
    22      0.9743     0.5954     1.4212    0.2500   0.3440   0.000100  (0.0s)
    23      0.9630     0.6532     1.3764    0.3500   0.4184   0.000100  (0.0s)
    24      0.9639     0.6590     1.2718    0.6000   0.6523   0.000100  (0.0s)
    25      0.9364     0.6763     1.2032    0.8000   0.7893   0.000100  (0.0s)
    26      0.9545     0.6358     1.1801    0.7500   0.7000   0.000100  (0.0s)
    27      0.8875     0.6879     1.1564    0.7000   0.6049   0.000100  (0.0s)


    28      0.8930     0.6474     1.2812    0.5000   0.5708   0.000100  (0.0s)
    29      0.8781     0.6763     1.3881    0.3500   0.4203   0.000100  (0.0s)
    30      0.8634     0.7283     1.3093    0.3000   0.3607   0.000100  (0.0s)
    31      0.8640     0.7052     1.2611    0.4500   0.4075   0.000100  (0.0s)
    32      0.8780     0.7225     1.2753    0.4500   0.4375   0.000100  (0.0s)
    33      0.7992     0.7514     1.3075    0.4000   0.4974   0.000100  (0.0s)
    34      0.8221     0.7110     1.3096    0.3500   0.4480   0.000100  (0.0s)
    35      0.7658     0.7457     1.3294    0.3500   0.3904   0.000050  (0.0s)
    36      0.7895     0.7283     1.3614    0.2000   0.2161   0.000050  (0.0s)
    37      0.7948     0.7110     1.3644    0.2000   0.2161   0.000050  (0.0s)
    38      0.7511     0.7572     1.3639    0.3000   0.4068   0.000050  (0.0s)


    39      0.7861     0.7457     1.3340    0.3000   0.3255   0.000050  (0.0s)
    40      0.8109     0.7110     1.3337    0.2000   0.2578   0.000050  (0.0s)

Early stopping at epoch 40. Best epoch: 25 (val_f1=0.7893)

Best: epoch 25, val_acc=0.8000, val_f1=0.7893
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe/jaffe_4c/FCNN_B1/model.pth
    Best w(CNN_TL)=0.00  val macro F1=0.7893
    Macro=0.4917  Micro=0.6500  Weighted=0.6150
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe/jaffe_4c_results.json


In [6]:
# RAF-DB
all_results['rafdb_7c'] = late_fusion_tl('rafdb', 7)
all_results['rafdb_4c'] = late_fusion_tl('rafdb', 4)


  Late Fusion TL — RAFDB 7c


  Train=10408  Val=1157  Test=2884


    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/7c/CNN_TL_B1/model.pth
    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/7c/FCNN_B1/model.pth


    Best w(CNN_TL)=0.45  val macro F1=0.7441
    Macro=0.7350  Micro=0.8294  Weighted=0.8231
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/rafdb_7c_results.json



  Late Fusion TL — RAFDB 4c


  Train=10408  Val=1157  Test=2884


    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/4c/CNN_TL_B1/model.pth
    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/4c/FCNN_B1/model.pth


    Best w(CNN_TL)=0.45  val macro F1=0.8469
    Macro=0.8322  Micro=0.8492  Weighted=0.8498
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/rafdb_4c_results.json


In [7]:
# KDEF
all_results['kdef_7c'] = late_fusion_tl('kdef', 7)
all_results['kdef_4c'] = late_fusion_tl('kdef', 4)


  Late Fusion TL — KDEF 7c


  Train=2631  Val=340  Test=337
    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/7c/CNN_TL_B1/model.pth


    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/7c/FCNN_B1/model.pth


    Best w(CNN_TL)=0.85  val macro F1=0.8899
    Macro=0.8358  Micro=0.8338  Weighted=0.8356
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/kdef_7c_results.json



  Late Fusion TL — KDEF 4c


  Train=2631  Val=340  Test=337
    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/4c/CNN_TL_B1/model.pth


    [SKIP train] loading existing /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/4c/FCNN_B1/model.pth


    Best w(CNN_TL)=0.50  val macro F1=0.9079
    Macro=0.9195  Micro=0.9318  Weighted=0.9301
    Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/kdef_4c_results.json


## Ringkasan Late Fusion TL (10 Benchmark Variants)

In [8]:
print(f"\n{'='*82}")
print(f'  Late Fusion TL — Summary across 10 benchmark variants')
print(f"{'='*82}")
print(f"  {'Dataset':<14} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Acc':>10} {'w(CNN_TL)':>10}")
print(f"  {'-'*72}")
for key, r in all_results.items():
    print(f"  {key:<14} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f} {r.get('best_cnn_tl_weight', 0):>10.2f}")


  Late Fusion TL — Summary across 10 benchmark variants
  Dataset             Macro      Micro   Weighted        Acc  w(CNN_TL)
  ------------------------------------------------------------------------
  primer_7c          0.2851     0.8267     0.8278     0.8267       0.25
  primer_4c          0.4723     0.7772     0.7787     0.7772       0.40
  ckplus_7c          0.8352     0.8814     0.8905     0.8814       1.00
  ckplus_4c          0.6039     0.8065     0.8034     0.8065       0.90
  jaffe_7c           0.1457     0.2000     0.1196     0.2000       0.05
  jaffe_4c           0.4917     0.6500     0.6150     0.6500       0.00
  rafdb_7c           0.7350     0.8294     0.8231     0.8294       0.45
  rafdb_4c           0.8322     0.8492     0.8498     0.8492       0.45
  kdef_7c            0.8358     0.8338     0.8356     0.8338       0.85
  kdef_4c            0.9195     0.9318     0.9301     0.9318       0.50
